***

*Course:* [Math 535](https://people.math.wisc.edu/~roch/mmids/) - Mathematical Methods in Data Science (MMiDS)  
*Chapter:* 3-Singular value decomposition   
*Author:* [Sebastien Roch](https://people.math.wisc.edu/~roch/), Department of Mathematics, University of Wisconsin-Madison  
*Updated:* Jan 27, 2024   
*Copyright:* &copy; 2024 Sebastien Roch

***

In [ ]:
# FOR e-BOOK ONLY
import os, sys
sys.path.insert(0, os.path.abspath('../../utils')) # use directory to mmids.py

In [ ]:
# PYTHON 3
import numpy as np
from numpy import linalg as LA
import matplotlib.pyplot as plt
import pandas as pd
import mmids
seed = 535
rng = np.random.default_rng(seed)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# FOR TeX ONLY
#plt.rcParams['figure.figsize'] = [4.00, 2.25] # for high-def figs
#plt.rcParams['figure.dpi'] = 200 # for high-def figs

$\newcommand{\horz}{\rule[.5ex]{2.5ex}{0.5pt}}$

## Approximating subspaces and the SVD

In this section, we introduce the singular value decomposition (SVD). We motivate it via the problem of finding a best approximating subspace to a collection of data points -- although it has applications far beyond.

### An objective, an algorithm, and a guarantee

Let $\boldsymbol{\alpha}_1,\dots,\boldsymbol{\alpha}_n$ be a collection of $n$ data points in $\mathbb{R}^m$. A natural way to extract low-dimensional structure in this dataset is to find a low-dimensional linear subspace $\mathcal{Z}$ of $\mathbb{R}^m$ such that the $\boldsymbol{\alpha}_i$'s are "close to it."

**Figure:** The closest line to some data points ([Source](https://web.stanford.edu/class/bios221/book/07-chap.html#a-line-that-minimizes-distances-in-both-directions))

![PCA](https://web.stanford.edu/class/bios221/book/07-chap_files/figure-html/fig-PCAmin-1.png)

$\bowtie$

**Mathematical formulation of the problem** Again the squared Euclidean norm turns out to be computationally convenient. So we look for a linear subspace $\mathcal{Z}$ that minimizes

$$
\sum_{i=1}^n \|\boldsymbol{\alpha}_i - \mathrm{proj}_\mathcal{Z}(\boldsymbol{\alpha}_i)\|^2 
$$

over all linear subspaces of $\mathbb{R}^m$ of dimension $k$. To solve this problem, which we refer to as the best approximating subspace problem, we make a series of observations.

The first observation gives a related, useful characterization of the optimal solution.

**LEMMA** **(Best Subspace as Maximimization)** Let $\boldsymbol{\alpha}_i$, $i =1\ldots,n$, be vectors in $\mathbb{R}^m$. A linear subspace $\mathcal{Z}$ of $\mathbb{R}^m$ that minimizes

$$
\sum_{i=1}^n \|\boldsymbol{\alpha}_i - \mathrm{proj}_\mathcal{Z}(\boldsymbol{\alpha}_i)\|^2 
$$

over all linear subspaces of dimension at most $k$ also maximizes

$$
\sum_{i=1}^n \|\mathrm{proj}_\mathcal{Z}(\boldsymbol{\alpha}_i)\|^2 
$$

over the same linear subspaces. And vice versa. $\flat$

*Proof idea:* This is a straightforward application of the triangle inequality.

**Figure:** An orthogonal projection ([Source](https://commons.wikimedia.org/wiki/File:Gram–Schmidt_process.svg))

![Orthogonal Projection](https://upload.wikimedia.org/wikipedia/commons/thumb/9/97/Gram–Schmidt_process.svg/640px-Gram–Schmidt_process.svg.png)

$\bowtie$

<!--TEXT

![Orthogonal Projection ([Source](https://commons.wikimedia.org/wiki/File:Gram–Schmidt_process.svg))](./figs/2560px-Gram–Schmidt_process.svg.png)

-->

*Proof:* By *Pythagoras*, 

$$
\|\boldsymbol{\alpha}_i - \mathrm{proj}_{\mathcal{Z}}(\boldsymbol{\alpha}_i)\|^2 + \|\mathrm{proj}_{\mathcal{Z}}(\boldsymbol{\alpha}_i)\|^2 
=  \|\boldsymbol{\alpha}_i\|^2
$$

since, by the *Orthogonal Projection Theorem*, $\mathrm{proj}_{\mathcal{Z}}(\boldsymbol{\alpha}_i)$ is orthogonal to $\boldsymbol{\alpha}_i - \mathrm{proj}_{\mathcal{Z}}(\boldsymbol{\alpha}_i)$. Rearranging, 

$$
\|\boldsymbol{\alpha}_i - \mathrm{proj}_{\mathcal{Z}}(\boldsymbol{\alpha}_i)\|^2
=  \|\boldsymbol{\alpha}_i\|^2 - \|\mathrm{proj}_{\mathcal{Z}}(\boldsymbol{\alpha}_i)\|^2.
$$

The result follows from the fact that the first term on the right-hand side does not depend on the choice of $\mathcal{Z}$. More specifically, optimizing over linear subspaces $\mathcal{Z}$ of dimension $k$,

\begin{align*}
\min_{\mathcal{Z}} \sum_{i=1}^n \|\boldsymbol{\alpha}_i - \mathrm{proj}_\mathcal{Z}(\boldsymbol{\alpha}_i)\|^2
&= \min_{\mathcal{Z}} \sum_{i=1}^n \left\{\|\boldsymbol{\alpha}_i\|^2 - \|\mathrm{proj}_{\mathcal{Z}}(\boldsymbol{\alpha}_i)\|^2\right\}\\
&= \sum_{i=1}^n \|\boldsymbol{\alpha}_i\|^2 + \min_{\mathcal{Z}} \left\{- \sum_{i=1}^n\|\mathrm{proj}_{\mathcal{Z}}(\boldsymbol{\alpha}_i)\|^2\right\}\\
&= \sum_{i=1}^n \|\boldsymbol{\alpha}_i\|^2 - \max_{\mathcal{Z}} \sum_{i=1}^n \|\mathrm{proj}_{\mathcal{Z}}(\boldsymbol{\alpha}_i)\|^2
\end{align*}.

$\square$

How do we specify a $k$-dimensional linear subspace? Through a basis of it, or - even better - an orthonormal basis. In the latter case, we also have an explicit formula for the orthogonal projection. And the dimension of the linear subspace is captured by the number of elements in the basis, by the *Dimension Theorem*. In other words, the best approximating subspace can be obtained by solving the problem 

$$
\max_{\mathbf{w}_1,\ldots,\mathbf{w}_k} \sum_{i=1}^n \left\|\sum_{j=1}^k \langle \boldsymbol{\alpha}_i, \mathbf{w}_j \rangle \,\mathbf{w}_j\right\|^2
$$

over all orthonormal lists $\mathbf{w}_1,\ldots,\mathbf{w}_k$ of length $k$. Our next observation rewrites the problem in matrix form. 

**LEMMA** **(Best Subpace in Matrix Form)** Consider the matrix $A \in \mathbb{R}^{n \times m}$ with rows $\boldsymbol{\alpha}_i^T$. A solution to the best approximating subspace problem is obtained by solving

$$
\max_{\mathbf{w}_1,\ldots,\mathbf{w}_k} \sum_{j=1}^k \|A \mathbf{w}_j\|^2
$$

over all orthonormal lists $\mathbf{w}_1,\ldots,\mathbf{w}_k$ of length $k$. $\flat$

*Proof idea:* We start with the one-dimensional case. A one-dimensional space $\mathcal{Z}$ is determined by a unit vector $\mathbf{w}_1$. The projection $\boldsymbol{\alpha}_i$ onto the span of $\mathbf{w}_1$ is given by the inner product formula $\langle \boldsymbol{\alpha}_i, \mathbf{w}_1 \rangle \,\mathbf{w}_1$. So 

\begin{align*}
\sum_{i=1}^n \|\langle \boldsymbol{\alpha}_i, \mathbf{w}_1 \rangle \,\mathbf{w}_1 \|^2
&= \sum_{i=1}^n \langle \boldsymbol{\alpha}_i, \mathbf{w}_1 \rangle ^2\\
&= \sum_{i=1}^n (\boldsymbol{\alpha}_i^T \mathbf{w}_1)^2\\
&= \|A \mathbf{w}_1\|^2
\end{align*}

where, again, $A$ is the matrix with rows $\boldsymbol{\alpha}_i ^T$, $i=1,\ldots, n$. Hence the solution to the one-dimensional problem is

$$
\mathbf{v}_1 \in \arg\max\{\|A \mathbf{w}_1\|:\|\mathbf{w}_1\| = 1\}.
$$

Here $\arg\max$ means that $\mathbf{v}_1$ is a vector $\mathbf{w}_1$ that achieves the maximum. Note that there could be more than one such $\mathbf{w}_1$, so the right-hand side is a set containing all such solutions. By the *Extreme Value Theorem* (proof omitted), there is at least one solution. Here we pick an arbitrary one.

*Proof:* For general $k$, we are looking for an orthonormal list $\mathbf{w}_1,\ldots,\mathbf{w}_k$ of length $k$ that maximizes

\begin{align*}
\sum_{i=1}^n \left\|\sum_{j=1}^k \langle \boldsymbol{\alpha}_i, \mathbf{w}_j \rangle \,\mathbf{w}_j\right\|^2
&= \sum_{i=1}^n \sum_{j=1}^k \langle \boldsymbol{\alpha}_i, \mathbf{w}_j \rangle ^2\\
&= \sum_{j=1}^k \left(\sum_{i=1}^n  (\boldsymbol{\alpha}_i^T \mathbf{w}_j)^2\right)\\
&= \sum_{j=1}^k \|A \mathbf{w}_j\|^2
\end{align*}

where $\mathcal{Z}$ is the subspace spanned by $\mathbf{w}_1,\ldots,\mathbf{w}_k$. On the second line, we used the *Properties of Orthonormal Lists*. That proves the claim. $\square$

We show next that a simple algorithm solves this problem.

**A greedy algorithm** Remarkably, the problem admits a greedy solution. Before discussing this solution, we take a small detour and give a classical example. Indeed, [greedy approaches](https://en.wikipedia.org/wiki/Greedy_algorithm) are a standard algorithmic tool for optimization problems. This is how Wikipedia describes them:

> A greedy algorithm is any algorithm that follows the problem-solving heuristic of making the locally optimal choice at each stage.[1] In many problems, a greedy strategy does not produce an optimal solution, but a greedy heuristic can yield locally optimal solutions that approximate a globally optimal solution in a reasonable amount of time.

**EXAMPLE:** Suppose you are thief and you broke into an antique shop at night. You cannot steal every item in the store. You have estimated that you can carry 10 lbs worth of merchandise, and still run fast enough to get away. Suppose that there are 4 items of interest with the following weights and values

| Item | Weight (lbs) | Value ($) |
| :-: | :-: | :-: |
| 1 | 8 | 1600 |
| 2 | 6 | 1100 |
| 3 | 4 | 700 |
| 4 | 1 | 100 |

There is exactly one of each item. Which items do you take? The siren is blaring, and you cannot try every combination. A quick scheme is to first pick the item of greatest value, i.e., Item 1. Now your bag has 8 lbs of merchandise in it. Then you consider the remaining items and choose whichever has highest value among those that still fit, i.e., those that are 2 lbs or lighter. That leaves only one choice, Item 4. Then you go - with a total value of 1700. 

This is called a greedy or myopic strategy, because you chose the first item to maximize your profit without worrying about the constraints it imposes on future choice. Indeed, in this case, there is a better combination: you could have picked Items 2 and 3 with a total value of 1800.

Other greedy schemes are possible here. A slightly more clever approach is to choose items of high *value per unit weight*, rather than considering value alone. But, that would not make a difference in this particular example (try it!). $\lhd$

Going back to the approximating subspace problem, we derive a greedy solution for it. Recall that we are looking for a solution to 

$$
\max_{\mathbf{w}_1,\ldots,\mathbf{w}_k} \|A \mathbf{w}_1\|^2 + \|A \mathbf{w}_2\|^2 + \cdots + \|A \mathbf{w}_1\|^2
$$

over all orthonormal lists $\mathbf{w}_1,\ldots,\mathbf{w}_k$ of length $k$.

In a greedy approach, we first solve for $\mathbf{w}_1$ by itself, without worrying about constraints it will impose on the next steps. That is, we compute

$$
\mathbf{v}_1 \in \arg\max\{\|A \mathbf{w}_1\|^2:\|\mathbf{w}_1\| = 1\}.
$$

As indicated before, by the *Extreme Value Theorem*, it can be shown that such a $\mathbf{v}_1$ exists, but may not be unique  (proof omitted). Then, fixing $\mathbf{w}_1 = \mathbf{v}_1$, we consider all unit vectors $\mathbf{w}_2$ orthogonal to $\mathbf{v}_1$ and maximize the contribution of $\mathbf{w}_2$ to the objective function. That is, we solve

$$
\mathbf{v}_2\in \arg\max
\{\|A \mathbf{w}_2\|^2 :\|\mathbf{w}_2\| = 1,\ \langle \mathbf{w}_2, \mathbf{v}_1 \rangle = 0\}.
$$

Again, such a $\mathbf{v}_2$ exists by the *Extreme Value Theorem*. Then proceeding by induction, for each $i = 3, \ldots, k$, we compute

$$
\mathbf{v}_i\in \arg\max
\{\|A \mathbf{w}_i\|^2 :\|\mathbf{w}_i\| = 1, \ \langle \mathbf{w}_i, \mathbf{v}_j \rangle = 0, \forall j \leq i-1\}.
$$

While it is clear that, after $k$ steps, this procedure constructs an orthonormal set of size $k$, it is far from obvious that it maximizes $\sum_{j=1}^k \|A \mathbf{v}_j\|^2$ over all such sets. Remarkably it does. The claim - which requires a proof - is that the best $k$-dimensional approximating subspace is obtained by finding the best $1$-dimensional subspace, then the best $1$-dimensional subspace orthogonal to the first one, and so on.  This follows from the next theorem. (Its proof is in the following (optional) subsection.)

**THEOREM** **(Greedy Finds Best Subspace)** Let $A \in \mathbb{R}^{n \times m}$ be a matrix with rows $\boldsymbol{\alpha}_i^T$, $i=1,\ldots,n$. For any $k \leq m$, let $\mathbf{v}_1,\ldots,\mathbf{v}_k$ be the greedy sequence constructed above. Then $\mathcal{Z}^* = \mathrm{span}(\mathbf{v}_1,\ldots,\mathbf{v}_k)$ is a solution to the minimization problem

$$
\min \left\{
\sum_{i=1}^n \|\boldsymbol{\alpha}_i - \mathrm{proj}_{\mathcal{Z}}(\boldsymbol{\alpha}_i)\|^2\ :\ \text{$\mathcal{Z}$ is a linear subspace of dimension $k$} 
\right\}.
$$

$\sharp$

Beyond the potential computational advantages of solving several one-dimensional problems rather one larger one, the greedy sequence has a more subtle property that is powerful. It allows us to solve the problem for all choices $k$ of target dimension *simultaneously*. To explain, note that the largest $k$ value, i.e. $k=m$, leads to a trivial problem. Indeed, the data points $\boldsymbol{\alpha}_i$, $i=1,\ldots,n$, already lie in an $m$-dimensional linear subspace, $\mathbb{R}^m$ itself. So we can take $\mathcal{Z} = \mathbb{R}^m$, and we have an objective value of

$$
\sum_{i=1}^n \|\boldsymbol{\alpha}_i - \mathrm{proj}_{\mathcal{Z}}(\boldsymbol{\alpha}_i)\|^2
= \sum_{i=1}^n \|\boldsymbol{\alpha}_i - \boldsymbol{\alpha}_i\|^2
= 0,
$$

which clearly cannot be improved. So any orthonormal basis of $\mathbb{R}^m$ will do. Say $\mathbf{e}_1,\ldots,\mathbf{e}_m$.  

On the other hand, the greedy sequence $\mathbf{v}_1,\ldots,\mathbf{v}_m$ has a very special property. For any $k \leq m$, the *truncation* $\mathbf{v}_1,\ldots,\mathbf{v}_k$ solves the approximating subspace problem in $k$ dimensions. That follows immediately from the *Greedy Finds Best Subspace Theorem*. The basis $\mathbf{e}_1,\ldots,\mathbf{e}_m$ (or any old basis of $\mathbb{R}^m$ for that matter) does *not* have this property. The idea of truncation is very useful and plays an important role in many data science applications; we will come back to it later in this section and the next one.

Note that we have not entirely solved the best approximating subspace problem from a computational point of view, as we have not given an explicit procedure to construct a solution to the $1$-dimensional subproblems. We've only shown that the solutions exist and have the right properties. We will take care of computational issues in a section below.

### From approximating subspaces to the SVD and back

While solving the approximating subspace problem in the previous section, we derived the building blocks of a matrix factorization that has found many applications, the [singular value decomposition (SVD)](https://en.wikipedia.org/wiki/Singular_value_decomposition). In this section, we define the SVD formally. We describe a simple method to compute it in the next section, where we also return to the application to dimensionality reduction.

**Some matrix results** We start with a few useful observations.

*Diagonal matrices:* First recall that a matrix $D \in \mathbb{R}^{k \times r}$ is diagonal if its non-diagonal entries are zero. That is, $i \neq j$ implies that $D_{ij} =0$. Note that a diagonal matrix is not necessarily square and that the diagonal elements are allowed to be zero. 

Multiplying a matrix by a diagonal one has a very specific effect. Let $A \in \mathbb{R}^{n \times k}$ and
$B \in \mathbb{R}^{r \times m}$. We focus here on the case $k \geq r$. The matrix product $A D$ produces a matrix whose columns are linear combinations of the columns of $A$ where the coefficients are taken from the corresponding column of $D$. But the columns of $D$ have at most one nonzero elements, the diagonal one. So the columns of $AD$ are in fact multiples of the columns of $A$

$$
AD
= 
\begin{pmatrix}
| &  & | \\
d_{11} \mathbf{a}_1 & \ldots & d_{rr} \mathbf{a}_r \\
| &  & | 
\end{pmatrix}
$$

where $\mathbf{a}_1,\ldots,\mathbf{a}_k$ are the columns of $A$ and $d_{11},\ldots,d_{rr}$ are the diagonal elements of $D$.

Similarly, the rows of $D B$ are linear combinations of the rows of $B$ where the coefficients are taken from the corresponding row of $D$. The rows of $D$ have at most one nonzero elements, the diagonal one. In the case, $k \geq r$, rows $r+1,\ldots,k$ necessarily have only zero entries since there is no diagonal entry there. Hence the first $r$ rows of $DB$ are multiples of the rows of $B$ and the next $k-r$ are $\mathbf{0}$

$$
DB
= 
\begin{pmatrix}
\horz & d_{11} \mathbf{b}_1^T & \horz \\
& \vdots &\\
\horz & d_{rr} \mathbf{b}_r^T & \horz\\
\horz & \mathbf{0} & \horz\\
& \vdots &\\
\horz & \mathbf{0} & \horz
\end{pmatrix}
$$

where $\mathbf{b}_1^T,\ldots,\mathbf{b}_r^T$ are the rows of $B$.

**EXAMPLE:** The following special case will be useful later in this chapter. Suppose $D, F \in \mathbb{R}^{n \times n}$ are both square diagonal matrices. Then $D F$ is the matrix whose diagonal elements are $d_{11} f_{11}, \ldots,d_{nn} f_{nn}$. $\lhd$ 

*Rank-one matrices:* Let $\mathbf{u} = (u_1,\ldots,u_n)^T \in \mathbb{R}^n$ and $\mathbf{v} = (v_1,\ldots,v_m)^T \in \mathbf{R}^m$ be two column (nonzero) vectors. Their outer product is defined as the matrix

$$
\mathbf{u} \mathbf{v}^T
= 
\begin{pmatrix}
u_1 v_1 & u_1 v_2 & \cdots & u_1 v_m\\
u_2 v_1 & u_2 v_2 & \cdots & u_2 v_m\\
\vdots & \vdots & \ddots & \vdots\\
u_n v_1 & u_n v_2 & \cdots & u_n v_m
\end{pmatrix}
= 
\begin{pmatrix}
| &  & | \\
v_{1} \mathbf{u} & \ldots & v_{m} \mathbf{u} \\
| &  & | 
\end{pmatrix}.
$$

This is not to be confused with the inner product $\mathbf{u}^T \mathbf{v}$, which requires $n = m$ and produces a scalar. 

If $\mathbf{u}$ and $\mathbf{v}$ are nonzero, the matrix $\mathbf{u} \mathbf{v}^T$ has rank one. Indeed, its columns are all a multiple of the same vector $\mathbf{u}$. So the column space spanned by the columns of $\mathbf{u} \mathbf{v}^T$ is one-dimensional. Vice versa, any rank-one matrix can be written in this form by definition of the rank.

*Matrix-matrix products and outer products:* We have seen many different interpretations of matrix-matrix products. Here is yet another one. Let $A = (a_{ij})_{i,j} \in \mathbb{R}^{n \times k}$ and
$B = (b_{ij})_{i,j} \in \mathbb{R}^{k \times m}$. Denote by $\mathbf{a}_1,\ldots,\mathbf{a}_k$ the columns of $A$
and denote by $\mathbf{b}_1^T,\ldots,\mathbf{b}_k^T$ the rows of $B$.

Then

\begin{align*}
A B
&=
\begin{pmatrix}
\sum_{j=1}^k a_{1j} b_{j1} & \sum_{j=1}^k a_{1j} b_{j2} & \cdots & \sum_{j=1}^k a_{1j} b_{jm}\\
\sum_{j=1}^k a_{2j} b_{j1} & \sum_{j=1}^k a_{2j} b_{j2} & \cdots & \sum_{j=1}^k a_{2j} b_{jm}\\
\vdots & \vdots & \ddots & \vdots\\
\sum_{j=1}^k a_{nj} b_{j1} & \sum_{j=1}^k a_{nj} b_{j2} & \cdots & \sum_{j=1}^k a_{nj} b_{jm}
\end{pmatrix}\\
&=
\sum_{j=1}^k
\begin{pmatrix}
 a_{1j} b_{j1} & a_{1j} b_{j2} & \cdots & a_{1j} b_{jm}\\
a_{2j} b_{j1} & a_{2j} b_{j2} & \cdots & a_{2j} b_{jm}\\
\vdots & \vdots & \ddots & \vdots\\
a_{nj} b_{j1} & a_{nj} b_{j2} & \cdots & a_{nj} b_{jm}
\end{pmatrix}\\
&=
\sum_{j=1}^k \mathbf{a}_j \mathbf{b}_j^T.
\end{align*}

In words, the matrix product $AB$ can be interpreted as a sum of $k$ rank-one matrices, each of which is the outer product of a column of $A$ with the corresponding row of $B$.

Because the rank of a sum is at most the sum of the ranks (as shown in a previous example), it follows that the rank of $AB$ is at most $k$. This is consistent with the fact that the rank of a product is at most the minimum of the ranks (also shown in a previous example).

**NUMERICAL CORNER:** In Numpy, the outer product is computed using [`numpy.outer`](https://numpy.org/doc/stable/reference/generated/numpy.outer.html).

In [ ]:
u = np.array([0., 2., -1.])
v = np.array([3., -2.])
Z = np.outer(u, v)
print(Z)

In [ ]:
print(LA.matrix_rank(Z))

$\unlhd$

**Definition and existence of the SVD** We now come to our main definition.

**DEFINITION** **(Singular Value Decomposition)** Let $A \in \mathbb{R}^{n\times m}$ be a matrix. A singular value decomposition (SVD) of $A$ is a matrix factorization

$$
A = U \Sigma V^T = \sum_{j=1}^r \sigma_j \mathbf{u}_j \mathbf{v}_j^T
$$

where the columns of $U \in \mathbb{R}^{n \times r}$ and those of $V \in \mathbb{R}^{m \times r}$ are orthonormal, and $\Sigma \in \mathbb{R}^{r \times r}$ is a diagonal matrix. Here the $\mathbf{u}_j$'s are the columns of $U$ and are referred to as left singular vectors. Similarly the $\mathbf{v}_j$'s are the columns of $V$ and are referred to as right singular vectors. The $\sigma_j$'s, which are positive and in decreasing order, i.e., 

$$
\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_r > 0,
$$ 

are the diagonal elements of $\Sigma$ and are referred to as singular values. $\natural$

To see where the equality $U \Sigma V^T = \sum_{j=1}^r \sigma_j \mathbf{u}_j \mathbf{v}_j^T$ comes from, 
we break it up into two steps. 

1. First note that, by the previous subsection, the matrix product $U \Sigma$ has columns $\sigma_1 \mathbf{u}_1,\ldots,\sigma_r \mathbf{u}_r$. 

2. The rows of $V^T$ are the columns of $V$ transposed.

3. In terms of outer products, the matrix product $U \Sigma V^T = (U \Sigma) V^T$ is the sum of the outer products of the columns of $U \Sigma$ and of the rows of $V^T$ (i.e., the columns of $V$ as row vectors).

That proves the equality.

Remarkably, any matrix has an SVD.

**THEOREM** **(Existence of SVD)** Any matrix $A \in \mathbb{R}^{n\times m}$ has a singular value decomposition. $\sharp$

*Proof idea:* We have already done most of the work. The proof works as follows: 

(1) Compute the greedy sequence $\mathbf{v}_1,\ldots,\mathbf{v}_r$ from the *Greedy Finds Best Subspace Theorem* until the largest $r$ such that 

$$
\|A \mathbf{v}_r\|^2 > 0
$$

or, otherwise, until $r=m$. The $\mathbf{v}_j$'s are orthonormal by construction.

(2) For $j=1,\ldots,r$, let

$$
\sigma_j = \|A \mathbf{v}_j\| \quad\text{and}\quad 
\mathbf{u}_j = \frac{1}{\sigma_j} A \mathbf{v}_j.
$$

Observe that, by our choice of $r$, the $\sigma_j$'s are $> 0$. They are also non-increasing: by definition of the greedy sequence,

$$
\sigma_i^2 = \max
\{\|A \mathbf{w}_i\|^2 :\|\mathbf{w}_i\| = 1, \ \langle \mathbf{w}_i, \mathbf{v}_j \rangle = 0, \forall j \leq i-1\},
$$

where the set of orthogonality constraints gets larger as $i$ increases. Hence, the $\mathbf{u}_j$'s have unit norm by definition. We show in an (optional) subsection below that they are also orthogonal. (An easier proof is relegated to a later section on the connection between SVD and a certain eigenvector decomposition.)

(3) Let $\mathbf{z} \in \mathbb{R}^m$ be any vector. To show that our construction is correct, we prove that $A \mathbf{z} = \left(U \Sigma V^T\right)\mathbf{z}$. Let $\mathcal{V} = \mathrm{span}(\mathbf{v}_1,\ldots,\mathbf{v}_r)$ and decompose $\mathbf{z}$ into orthogonal components 

$$
\mathbf{z}
= \mathrm{proj}_{\mathcal{V}}(\mathbf{z}) + (\mathbf{z} - \mathrm{proj}_{\mathcal{V}}(\mathbf{z}))
= \sum_{j=1}^r \langle \mathbf{z}, \mathbf{v}_j\rangle \,\mathbf{v}_j
+ (\mathbf{z} - \mathrm{proj}_{\mathcal{V}}(\mathbf{z})).
$$

Applying $A$ and using linearity, we get

\begin{align*}
A \mathbf{z}
&= \sum_{j=1}^r \langle \mathbf{z}, \mathbf{v}_j\rangle \, A\mathbf{v}_j
+ A (\mathbf{z} - \mathrm{proj}_{\mathcal{V}}(\mathbf{z})).
\end{align*}

We claim that $A (\mathbf{z} - \mathrm{proj}_{\mathcal{V}}(\mathbf{z})) = \mathbf{0}$. If $\mathbf{z} - \mathrm{proj}_{\mathcal{V}}(\mathbf{z}) = \mathbf{0}$, that's certainly the case. If not, let

$$
\mathbf{w} = \frac{\mathbf{z} - \mathrm{proj}_{\mathcal{V}}(\mathbf{z})}{\|\mathbf{z} - \mathrm{proj}_{\mathcal{V}}(\mathbf{z})\|}.
$$

By definition of $r$, 

$$
0 = \max
\{\|A \mathbf{w}_{r+1}\|^2 :\|\mathbf{w}_{r+1}\| = 1, \ \langle \mathbf{w}_{r+1}, \mathbf{v}_j \rangle = 0, \forall j \leq r\}.
$$

Put differently, $\|A \mathbf{w}_{r+1}\|^2  = 0$ (i.e. $A \mathbf{w}_{r+1} = \mathbf{0}$), for any unit vector $\mathbf{w}_{r+1}$ orthogonal to $\mathbf{v}_1,\ldots,\mathbf{v}_r$. That applied in particular to $\mathbf{w}_{r+1} = \mathbf{w}$ by the *Orthogonal Projection Theorem*.

Hence, using the definition of $\mathbf{u}_j$ and $\sigma_j$, we get

\begin{align*}
A \mathbf{z}
&= \sum_{j=1}^r \langle \mathbf{z}, \mathbf{v}_j\rangle \, \sigma_j \mathbf{u}_j\\
&= \sum_{j=1}^r \sigma_j \mathbf{u}_j \mathbf{v}_j^T \mathbf{z}\\
&=\left(\sum_{j=1}^r \sigma_j \mathbf{u}_j \mathbf{v}_j^T\right)\mathbf{z}\\
&= \left(U \Sigma V^T\right)\mathbf{z}.
\end{align*}

That proves the existence of the SVD.

We record the following important consequence.

**LEMMA** **(SVD and Rank)** Let $A \in \mathbb{R}^{n\times m}$ have singular value decomposition $A = U \Sigma V^T$ with $U \in \mathbb{R}^{n \times r}$ and $V \in \mathbb{R}^{m \times r}$. Then the columns of $U$ form an orthonormal basis of $\mathrm{col}(A)$ and the columns of $V$ form an orthonormal basis of $\mathrm{row}(A)$. In particular, the rank of $A$ is $r$. $\flat$

*Proof idea:* We use the SVD to show that the span of the columns of $U$ is $\mathrm{col}(A)$, and similarly for $V$.

*Proof:* We first prove that any column of $A$ can be written as a linear combination of the columns of $U$. Indeed, this follows immediately from the SVD by noting that for any canonical basis vector $\mathbf{e}_i \in \mathbb{R}^m$ (which produces column $i$ of $A$ with $A \mathbf{e}_i$)

\begin{align*}
A \mathbf{e}_i
=\left(\sum_{j=1}^r \sigma_j \mathbf{u}_j \mathbf{v}_j^T\right)\mathbf{e}_i
= \sum_{j=1}^r (\sigma_j \mathbf{v}_j^T \mathbf{e}_i) \,\mathbf{u}_j.
\end{align*}

Vice versa, any column of $U$ can be written as a linear combination of the columns of $A$. To see this, we use the orthonormality of the $\mathbf{v}_j$'s and the positivity of the singular values to obtain

\begin{align*}
A (\sigma_i^{-1} \mathbf{v}_i)
= \sigma_i^{-1}\left(\sum_{j=1}^r \sigma_j \mathbf{u}_j \mathbf{v}_j^T\right)\mathbf{v}_i
= \sigma_i^{-1} \sum_{j=1}^r (\sigma_j \mathbf{v}_j^T \mathbf{v}_i) \,\mathbf{u}_j
= \sigma_i^{-1} (\sigma_i \mathbf{v}_i^T \mathbf{v}_i) \,\mathbf{u}_i
= \mathbf{u}_i.
\end{align*}

That is, $\mathrm{col}(U) = \mathrm{col}(A)$. We have already shown that the columns of $U$ are orthonormal. Since their span is $\mathrm{col}(A)$, they form an orthonormal basis of it. Applying the same argument to $A^T$ gives the claim for $V$ (try it!). $\square$

**EXAMPLE:** Let 

$$
A = \begin{pmatrix}
1 & 0\\
-1 & 0
\end{pmatrix}.
$$

We compute its SVD. We have not (yet) seen a general recipe to compute the SVD by hand. But in this case it can be done (or guessed) using what we know about the SVD. Note first that $A$ is not invertible. Indeed, its row are a multiple of one another. In particular, they are not linearly independent. In fact, that tells us that the rank of $A$ is $1$, the dimension of its row space. In the rank one case, computing the SVD boils down to writing the matrix $A$ in outer product form

$$
A = \sigma_1 \mathbf{u}_1 \mathbf{v}_1^T
$$

where we require that $\sigma_1 > 0$ and that $\mathbf{u}_1, \mathbf{v}_1$ are of unit norm. 

Recall that an outer product has columns that are all multiples of the same vector. Here because the second column of $A$ is $\mathbf{0}$, we must have that the second component of $\mathbf{v}_1$ is $0$. To be of unit norm, its first component must be $1$ or $-1$. (The choice here does not matter because multiply all left and right singular vectors by $-1$ produces another SVD.) We choose $1$, i.e., we let

$$
\mathbf{v}_1 
= \begin{pmatrix}
1\\
0
\end{pmatrix}.
$$

That vector is indeed an orthonormal basis of the row space of $A$. Then we need 

$$
\sigma_1 \mathbf{u}_1 = \begin{pmatrix}
1\\
-1
\end{pmatrix}.
$$

For $\mathbf{u}_1$ to be of unit norm, we must have

$$
\mathbf{u}_1 = \begin{pmatrix}
1/\sqrt{2}\\
-1/\sqrt{2}
\end{pmatrix}
\quad
\text{and}
\quad
\sigma_1 = \sqrt{2}.
$$

Observe that $\mathbf{u}_1$ is indeed an orthonormal basis of the column space of $A$. $\lhd$

We collect in the next lemma some relationships between the singular vectors and singular values that will be used repeatedly. The proof essentially follows previous calculations and is left as an exercise.

The second claim below will be the basis for a connection between the SVD and a certain eigenvector decomposition. We will come back to this useful connection in the next chapter.

**LEMMA** **(SVD Relations)** Let 
$A = \sum_{j=1}^r \sigma_j \mathbf{u}_j \mathbf{v}_j^T$
be an SVD of $A \in \mathbb{R}^{n \times m}$ with $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_r > 0$. Then, for $i=1,\ldots,r$,

$$
A \mathbf{v}_i = \sigma_i \mathbf{u}_i,
\qquad
A^T \mathbf{u}_i = \sigma_i \mathbf{v}_i,
\qquad
\|A \mathbf{v}_i\| = \sigma_i,
\qquad 
\|A^T \mathbf{u}_i\| = \sigma_i.
$$

Moreover

$$
A^T A \mathbf{v}_i = \sigma_i^2 \mathbf{v}_i,
\qquad 
A A^T \mathbf{u}_i = \sigma_i^2 \mathbf{u}_i.
$$

Finally, for $j \neq i$,

$$
\langle A \mathbf{v}_i, A \mathbf{v}_j \rangle = 0,
\qquad
\langle A^T \mathbf{u}_i, A^T \mathbf{u}_j \rangle = 0.
$$

$\flat$

**Full vs. compact SVD** What we have introduced above is in fact referred to as a [compact SVD](https://en.wikipedia.org/wiki/Singular_value_decomposition#Compact_SVD). In contrast, in a [full SVD](https://en.wikipedia.org/wiki/Singular_value_decomposition), the matrices $U$ and $V$ are square and orthogonal, and the matrix $\Sigma$ is diagonal, but may not be square and may have zeros on the diagonal. In other words, in that case, the columns of $U \in \mathbb{R}^{n \times n}$ form an orthonormal basis of $\mathbb{R}^n$ and the columns of $V \in \mathbb{R}^{m \times m}$ form an orthonormal basis of $\mathbb{R}^m$. An SVD in its full form is illustrated below.

**Figure:** SVD in full form ([Source](https://commons.wikimedia.org/wiki/File:Singular_value_decomposition_visualisation.svg))

![SVD](https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Singular_value_decomposition_visualisation.svg/412px-Singular_value_decomposition_visualisation.svg.png)

$\bowtie$

<!--TEX

![SVD in full form ([Source](https://commons.wikimedia.org/wiki/File:Singular_value_decomposition_visualisation.svg))](.figs/Singular_value_decomposition_visualisation.svg.png)

-->

Let $A = U_1 \Sigma_1 V_1^T$ be a compact SVD. Complete the columns of $U_1$ into an orthonormal basis of $\mathbb{R}^n$ and let $U_2$ be the matrix whose columns are the additional basis vectors. Similary, complete the columns of $V_1$ into an orthonormal basis of $\mathbb{R}^m$ and let $V_1$ be the matrix whose columns are the additional basis vectors. Then a full SVD is given by 

$$
U = \begin{pmatrix}
U_1 & U_2
\end{pmatrix}
\quad
V = \begin{pmatrix}
V_1 & V_2
\end{pmatrix}
\quad
\Sigma
= \begin{pmatrix}
\Sigma_1 & \mathbf{0}_{r \times (m-r)}\\
\mathbf{0}_{(n-r)\times r} & \mathbf{0}_{(n-r)\times (m-r)}
\end{pmatrix}.
$$

By the *SVD and Rank Lemma*, the columns of $U_1$ form an orthonormal basis of $\mathrm{col}(A)$. Because $\mathrm{col}(A)^\perp = \mathrm{null}(A^T)$, the columns of $U_2$ form an orthonormal basis of $\mathrm{null}(A^T)$. Similarly, the columns of $V_1$ form an orthonormal basis of $\mathrm{col}(A^T)$. Because $\mathrm{col}(A^T)^\perp = \mathrm{null}(A)$, the columns of $V_2$ form an orthonormal basis of $\mathrm{null}(A)$. Hence, a full SVD provides an orthonormal basis for all four fundamental subspaces of $A$.

The full SVD also has a natural geometric interpretation. To quote [Sol, p. 133]:

> The SVD provides a complete geometric characterization of the action of $A$. Since $U$ and $V$ are orthogonal, they have no effect on lengths and angles; as a diagonal matrix, $\Sigma$ scales individual coordinate axes. Since the SVD always exists, all matrices $A \in \mathbb{R}^{n \times m}$ are a composition of an isometry, a scale in each coordinate, and a second isometry. 

This sequence of operations is illustrated below.

**Figure:** Geometric interpretation of the SVD ([Source](https://commons.wikimedia.org/wiki/File:Singular-Value-Decomposition.svg))

![Geometric meaning](https://upload.wikimedia.org/wikipedia/commons/thumb/b/bb/Singular-Value-Decomposition.svg/531px-Singular-Value-Decomposition.svg.png)

$\bowtie$

<!--TEX

![Geometric interpretation of the SVD ([Source](https://commons.wikimedia.org/wiki/File:Singular-Value-Decomposition.svg))](./figs/Singular-Value-Decomposition.svg.png)

-->

Vice versa, given a full SVD $A = U \Sigma V^T$, the compact SVD can be obtained by keeping only the square submatrix of $\Sigma$ with stricly positive diagonal entries, together with the corresponding columns of $U$ and $V$.

A comparison of full (first row) and compact (third row) SVDs is illustrated below.

**Figure:** Different variants of the SVD ([Source](https://commons.wikimedia.org/wiki/File:Reduced_Singular_Value_Decompositions.svg))

![ReducedSVD](https://upload.wikimedia.org/wikipedia/commons/thumb/c/c4/Reduced_Singular_Value_Decompositions.svg/233px-Reduced_Singular_Value_Decompositions.svg.png)

$\bowtie$

<!--TEX

![Different variants of the SVD ([Source](https://commons.wikimedia.org/wiki/File:Reduced_Singular_Value_Decompositions.svg))](https://upload.wikimedia.org/wikipedia/commons/thumb/c/c4/Reduced_Singular_Value_Decompositions.svg/233px-Reduced_Singular_Value_Decompositions.svg.png)

--> 

**EXAMPLE:** **(continued)** Let again 

$$
A = \begin{pmatrix}
1 & 0\\
-1 & 0
\end{pmatrix}.
$$

We previously computed its compact SVD

$$
A = \sigma_1 \mathbf{u}_1 \mathbf{v}_1^T
$$

where

$$
\mathbf{u}_1 = \begin{pmatrix}
1/\sqrt{2}\\
-1/\sqrt{2}
\end{pmatrix},
\quad
\quad
\mathbf{v}_1 
= \begin{pmatrix}
1\\
0
\end{pmatrix},
\quad
\text{and}
\quad
\sigma_1 = \sqrt{2}.
$$

We now compute a full SVD. For this, we need to complete the bases. We can choose

$$
\mathbf{u}_2 = \begin{pmatrix}
1/\sqrt{2}\\
1/\sqrt{2}
\end{pmatrix},
\quad
\quad
\mathbf{v}_2 
= \begin{pmatrix}
0\\
1
\end{pmatrix},
\quad
\text{and}
\quad
\sigma_2 = 0.
$$

Then, a full SVD is given by

$$
U = \begin{pmatrix}
1/\sqrt{2} & 1/\sqrt{2}\\
-1/\sqrt{2} & 1/\sqrt{2}
\end{pmatrix},
\quad
\quad
V
= \begin{pmatrix}
1 & 0\\
0 & 1
\end{pmatrix},
\quad
\text{and}
\quad
\Sigma = \begin{pmatrix}
\sqrt{2} & 0\\
0 & 0
\end{pmatrix}.
$$

Indeed, $A = U \Sigma V^T$ (check it!). $\lhd$

**SVD and Approximating Subspace** In constructing the SVD of $A$, we used the greedy sequence for the best approximating subspace. Vice versa, given an SVD of $A$, we can read off the solution to the approximating subspace problem. In other words, there was nothing special about the specific construction we used to prove existence of the SVD. While a matrix may have many SVDs, they all give a solution to the approximating subspace problems. 

Further, this perspective gives what is known as a variational characterization of the singular values. We will have more to say about variational characterizations and their uses in the next chapter.

*SVD and greedy sequence* Indeed, let 

$$
A = \sum_{j=1}^r \sigma_j \mathbf{u}_j \mathbf{v}_j^T
$$

be an SVD of $A$ with

$$
\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_r > 0.
$$ 

We show that the $\mathbf{v}_j$'s form a greedy sequence for the approximating subspace problem. Complete $\mathbf{v}_1,\ldots,\mathbf{v}_r$ into an orthonormal basis of $\mathbb{R}^m$ by adding appropriate vectors $\mathbf{v}_{j+1},\ldots,\mathbf{v}_m$. By construction, $A\mathbf{v}_{i} = \mathbf{0}$ for all $i=j+1,\ldots,m$.

We start with the case $j=1$. For any unit vector $\mathbf{w} \in \mathbb{R}^m$, we expand it as $\mathbf{w} = \sum_{i=1}^m \langle \mathbf{w}, \mathbf{v}_i\rangle \,\mathbf{v}_i$. By the *Properties of Orthonormal Lists*,

\begin{align*}
\|A \mathbf{w}\|^2
&=
\left\|
A \left(
\sum_{i=1}^m \langle \mathbf{w}, \mathbf{v}_i\rangle \,\mathbf{v}_i
\right)
\right\|^2\\
&= 
\left\|
\sum_{i=1}^m \langle \mathbf{w}, \mathbf{v}_i\rangle \,A \mathbf{v}_i
\right\|^2\\
&= 
\left\|
\sum_{i=1}^r \langle \mathbf{w}, \mathbf{v}_i\rangle \,\sigma_i \mathbf{u}_i
\right\|^2\\
&= \sum_{i=1}^r \sigma_i^2 \langle \mathbf{w}, \mathbf{v}_i\rangle^2,
\end{align*}

where we used the orthonormality of the $\mathbf{u}_i$'s and the fact that $\mathbf{v}_{j+1},\ldots,\mathbf{v}_m$ are in the null space of $A$. Because the $\sigma_i$'s are non-increasing, this last sum is maximized by taking $\mathbf{w} = \mathbf{v}_1$. So we have shown that $\mathbf{v}_1 \in \arg\max\{\|A \mathbf{w}\|^2:\|\mathbf{w}\| = 1\}$.

By the *Properties of Orthonormal Lists*,

$$
\sum_{i=1}^m \langle \mathbf{w}, \mathbf{v}_i\rangle^2
= \left\|\sum_{i=1}^m \langle \mathbf{w}, \mathbf{v}_i\rangle \mathbf{v}_i \right\|^2
= \|\mathbf{w}\|^2 
= 1.
$$

Hence,
because the $\sigma_i$'s are non-increasing, the sum


$$
\|A \mathbf{w}\|^2
= \sum_{i=1}^r \sigma_i^2 \langle \mathbf{w}, \mathbf{v}_i\rangle^2
\leq \sigma_1^2.
$$

This upper bound is achieved by taking $\mathbf{w} = \mathbf{v}_1$. So we have shown that $\mathbf{v}_1 \in \arg\max\{\|A \mathbf{w}\|^2:\|\mathbf{w}\| = 1\}$.

More generally, for any unit vector $\mathbf{w} \in \mathbb{R}^m$ that is orthogonal to $\mathbf{v}_1,\ldots,\mathbf{v}_{j-1}$, we expand it as $\mathbf{w} = \sum_{i=j}^m \langle \mathbf{w}, \mathbf{v}_i\rangle \,\mathbf{v}_i$.
Then, as long as $j\leq r$,

\begin{align*}
\|A \mathbf{w}\|^2
&= 
\left\|
\sum_{i=j}^m \langle \mathbf{w}, \mathbf{v}_i\rangle \,A \mathbf{v}_i
\right\|^2\\
&= 
\left\|
\sum_{i=j}^r \langle \mathbf{w}, \mathbf{v}_i\rangle \,\sigma_i \mathbf{u}_i
\right\|^2\\
&= \sum_{i=j}^r \sigma_i^2 \langle \mathbf{w}, \mathbf{v}_i\rangle^2\\
&\leq \sigma_j^2.
\end{align*}

where again we used that the $\sigma_i$'s are non-increasing and $\sum_{i=1}^m \langle \mathbf{w}, \mathbf{v}_i\rangle^2 = 1$. This last bound is achieved by taking $\mathbf{w} = \mathbf{v}_j$.

So we have shown the following.

**THEOREM** **(Variational Characterization of Singular Values)** Let 
$A = \sum_{j=1}^r \sigma_j \mathbf{u}_j \mathbf{v}_j^T$ be an SVD of $A$ with
$\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_r > 0$. 
Then 

$$
\sigma_j^2 = \max
\{\|A \mathbf{w}\|^2:\|\mathbf{w}\| = 1, \ \langle \mathbf{w}, \mathbf{v}_i \rangle = 0, \forall i \leq j-1\},
$$

and

$$
\mathbf{v}_j\in \arg\max
\{\|A \mathbf{w}\|^2:\|\mathbf{w}\| = 1, \ \langle \mathbf{w}, \mathbf{v}_i \rangle = 0, \forall i \leq j-1\}.
$$

$\sharp$

*Solving the approximating subspace problem via SVD* Think of the rows $\boldsymbol{\alpha}_i^T$ of a matrix $A \in \mathbb{R}^{n \times m}$ as a collection of $n$ data points in $\mathbb{R}^m$. Let 

$$
A = \sum_{j=1}^r \sigma_j \mathbf{u}_j \mathbf{v}_j^T
$$

be a (compact) SVD of $A$. Fix $k \leq \mathrm{rk}(A)$. We are looking for a linear subspace $\mathcal{Z}$ that minimizes

$$
\sum_{i=1}^n \|\boldsymbol{\alpha}_i - \mathrm{proj}_\mathcal{Z}(\boldsymbol{\alpha}_i)\|^2 
$$

over all linear subspaces of $\mathbb{R}^m$ of dimension at most $k$. By the *Greedy Finds Best Subspace Theorem* and the observations above, a solution is given by

$$
\mathcal{Z}
= \mathrm{span}(\mathbf{v}_1,\ldots,\mathbf{v}_k).
$$

By the proofs of the *Best Subspace as Maximization* and *Best Subspace in Matrix Form* lemmas, the objective value achieved is

\begin{align*}
\sum_{i=1}^n \|\boldsymbol{\alpha}_i - \mathrm{proj}_\mathcal{Z}(\boldsymbol{\alpha}_i)\|^2 
&= \sum_{i=1}^n \|\boldsymbol{\alpha}_i\|^2 - \sum_{j=1}^k \|A\mathbf{v}_j\|^2\\
&= \sum_{i=1}^n \|\boldsymbol{\alpha}_i\|^2 - \sum_{j=1}^k \sigma_j^2.
\end{align*}

So the singular value $\sigma_j$ associated to right singular vector $\mathbf{v}_j$ captures its contribution to the fit of the approximating subspace. The larger the singular value, the larger the contribution.

We compute $\mathbf{z}_i := \mathrm{proj}_\mathcal{Z}(\boldsymbol{\alpha}_i)$ for each $i$ as follows (in row form)

\begin{align*}
\mathbf{z}_i^T
&= \sum_{j=1}^k \langle \boldsymbol{\alpha}_i, \mathbf{v}_j\rangle \,\mathbf{v}_j^T\\
&= \sum_{j=1}^k \boldsymbol{\alpha}_i^T \mathbf{v}_j \mathbf{v}_j^T\\
&= A_{i,\cdot} V_{(k)} V_{(k)}^T,
\end{align*}

where $V_{(k)}$ is the matrix with the first $k$ columns of $V$. Let $Z$ be the matrix with rows $\mathbf{z}_i^T$. Then we have

$$
Z = A V_{(k)} V_{(k)}^T = U_{(k)} \Sigma_{(k)} V_{(k)}^T
= \sum_{j=1}^k \sigma_j \mathbf{u}_j \mathbf{v}_j^T,
$$

where $U_{(k)}$ is the matrix with the first $k$ columns of $U$, and $\Sigma_{(k)}$ is the matrix with the first $k$ rows and columns of $\Sigma$. Indeed, recall that $A \mathbf{v}_j = \sigma_j \mathbf{u}_j$, or in matrix form $A V_{(k)} = U_{(k)} \Sigma_{(k)}$. The rightmost expression for $Z$ reveals that it is in fact a truncated SVD. We can interpret the rows of $U_{(k)} \Sigma_{(k)}$ as the coefficients of each data point in the basis $\mathbf{v}_1,\ldots,\mathbf{v}_k$.

We can re-write the objective function in a more compact matrix form by using the Frobenius norm as follows

\begin{align*}
\sum_{i=1}^n \|\boldsymbol{\alpha}_i - \mathrm{proj}_\mathcal{Z}(\boldsymbol{\alpha}_i)\|^2 
&= \sum_{i=1}^n \|\boldsymbol{\alpha}_i - \mathbf{z}_i\|^2
= \sum_{i=1}^n \|\boldsymbol{\alpha}_i^T - \mathbf{z}_i^T\|^2
= \|A - Z\|_F^2.
\end{align*}

We note that the matrix $Z$ has rank less or equal than $k$. Indeed, all of its rows lie in the optimal subspace $\mathcal{Z}$, which has dimension $k$ by construction. We will see later (in an optional section) that $Z$ is the best approximation to $A$ among all rank-$k$ matrices under the Frobenius norm, that is,

$$
\|A - Z\|_F
\leq \|A - B\|_F
$$

for any matrix $B$ of rank at most $k$.